# 🛍️ AI Retail Demand Forecasting Agent

Python starter notebook for exploring retail sales, building a baseline demand forecast, evaluating it, and deriving an illustrative reorder point. Uses synthetic daily data; replace it with validated sales history for real decisions.

## 🎯 Objectives
- Explore sales patterns
- Use a chronological train/test split
- Forecast with a moving-average baseline
- Evaluate MAE and RMSE
- Produce an illustrative inventory signal
- Extend with grounded LLM explanations

Forecasts are decision support, not guaranteed demand.

## 📦 Generate sample data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
np.random.seed(42)

dates = pd.date_range('2025-01-01', periods=240, freq='D')
trend = np.linspace(35, 55, len(dates))
weekly = 8 * np.sin(2 * np.pi * dates.dayofweek / 7)
seasonal = 6 * np.sin(2 * np.pi * np.arange(len(dates)) / 30)
noise = np.random.normal(0, 5, len(dates))
sales = np.maximum(0, np.round(trend + weekly + seasonal + noise)).astype(int)
df = pd.DataFrame({'date': dates, 'sku': 'SKU-001', 'units_sold': sales})
df.head()

In [ ]:
df.describe()
plt.figure(figsize=(12, 4))
plt.plot(df.date, df.units_sold)
plt.title('Daily units sold — sample SKU')
plt.xlabel('Date'); plt.ylabel('Units')
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

In [ ]:
# Hold out the most recent 30 days; do not shuffle time series.
test_days = 30
train, test = df.iloc[:-test_days].copy(), df.iloc[-test_days:].copy()
window = 14
baseline = train.units_sold.tail(window).mean()
test['forecast'] = baseline
mae = mean_absolute_error(test.units_sold, test.forecast)
rmse = np.sqrt(mean_squared_error(test.units_sold, test.forecast))
print(f'MAE: {mae:.2f} units | RMSE: {rmse:.2f} units')

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(train.date.tail(45), train.units_sold.tail(45), label='Recent actuals')
plt.plot(test.date, test.units_sold, label='Test actuals')
plt.plot(test.date, test.forecast, label='Baseline forecast')
plt.legend(); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

In [ ]:
# Simplified illustrative reorder point: lead-time demand + safety stock.
lead_time_days = 7
safety_stock_units = 20
avg_daily_forecast = float(test.forecast.mean())
reorder_point = int(np.ceil(avg_daily_forecast * lead_time_days + safety_stock_units))
print(f'Average forecast/day: {avg_daily_forecast:.1f}')
print(f'Illustrative reorder point: {reorder_point} units')

In [ ]:
def build_summary(mae_value, rmse_value, daily_forecast, reorder_point):
    return (f'Baseline demand is about {daily_forecast:.1f} units/day. '
            f'Holdout MAE={mae_value:.1f}, RMSE={rmse_value:.1f}. '
            f'Illustrative reorder point={reorder_point} units. '
            'Review uncertainty and business constraints before acting.')

print(build_summary(mae, rmse, avg_daily_forecast, reorder_point))

## 🌐 Optional GenAI extension
Use an LLM to explain computed metrics or summarize anomalies. Ground responses in measured data; do not let the model invent sales, prices, stock, or lead times. Store API keys in environment variables.

## 🗺️ Production roadmap
- Load POS/ERP data from CSV, SQL, or approved APIs
- Support multiple SKUs, stores, promotions, holidays, and stockouts
- Compare seasonal-naive, statistical, and ML models
- Backtest across multiple forecast origins
- Track WAPE, MAE, and forecast bias by SKU
- Simulate reorder policies with human review
- Add Streamlit dashboard, data checks, and model monitoring